# YOLO Fine-Tuning for Roof Detection

This notebook trains a pre-trained YOLO model using roof images with fine-tuning. The dataset contains roof images with labeled bounding boxes in YOLO format.

## 1. Import Required Libraries

In [1]:
import os
import sys
import shutil
from pathlib import Path
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import yaml

# Import YOLO from ultralytics
from ultralytics import YOLO

# Set random seeds for reproducibility
np.random.seed(42)

print("Libraries imported successfully!")

Libraries imported successfully!


## 2. Setup YOLO Environment and Paths

In [2]:
# Define paths
NOTEBOOK_DIR = Path.cwd()
DATA_SOURCE_DIR = NOTEBOOK_DIR / "data" / "roofs"
DATASET_DIR = NOTEBOOK_DIR / "roof_dataset_yolo"
RUNS_DIR = NOTEBOOK_DIR / "runs"

# Create directories if they don't exist
DATASET_DIR.mkdir(exist_ok=True)
RUNS_DIR.mkdir(exist_ok=True)

print(f"Notebook directory: {NOTEBOOK_DIR}")
print(f"Data source directory: {DATA_SOURCE_DIR}")
print(f"Dataset directory: {DATASET_DIR}")
print(f"Runs directory: {RUNS_DIR}")

# Check if source data exists
if not DATA_SOURCE_DIR.exists():
    print(f"Error: Data source directory not found: {DATA_SOURCE_DIR}")
else:
    # List available files
    image_files = list(DATA_SOURCE_DIR.glob("*.png"))
    label_files = list(DATA_SOURCE_DIR.glob("*.txt"))
    print(f"\nFound {len(image_files)} images and {len(label_files) - 1} label files (excluding classes.txt)")
    print(f"Sample files: {image_files[:3] if image_files else 'None'}")

Notebook directory: /home/jovyan/work
Data source directory: /home/jovyan/work/data/roofs
Dataset directory: /home/jovyan/work/roof_dataset_yolo
Runs directory: /home/jovyan/work/runs

Found 200 images and 200 label files (excluding classes.txt)
Sample files: [PosixPath('/home/jovyan/work/data/roofs/staticmap (1).png'), PosixPath('/home/jovyan/work/data/roofs/staticmap (10).png'), PosixPath('/home/jovyan/work/data/roofs/staticmap (100).png')]


## 3. Load and Validate Dataset

In [3]:
def validate_yolo_label(label_path):
    """
    Validate if a label file is in YOLO format
    Expected format: class_id x_center y_center width height (all normalized to 0-1)
    """
    try:
        with open(label_path, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            parts = line.strip().split()
            if len(parts) != 5:
                return False, f"Expected 5 values, got {len(parts)}"
            
            class_id = int(parts[0])
            coords = [float(x) for x in parts[1:]]
            
            # Check if coordinates are normalized (0-1)
            for coord in coords:
                if not (0 <= coord <= 1):
                    return False, f"Coordinate {coord} out of range [0, 1]"
        
        return True, "Valid YOLO format"
    except Exception as e:
        return False, str(e)

# Collect all image-label pairs
data_pairs = []
image_files = sorted(DATA_SOURCE_DIR.glob("*.png"))

print(f"Validating {len(image_files)} image-label pairs...")

valid_count = 0
invalid_count = 0

for img_path in image_files:
    label_path = img_path.with_suffix('.txt')
    
    if label_path.exists() and label_path.name != 'classes.txt':
        is_valid, msg = validate_yolo_label(label_path)
        if is_valid:
            data_pairs.append((img_path, label_path))
            valid_count += 1
        else:
            print(f"Invalid: {img_path.name} - {msg}")
            invalid_count += 1

print(f"\n✓ Valid pairs: {valid_count}")
print(f"✗ Invalid pairs: {invalid_count}")
print(f"Total dataset size: {len(data_pairs)} samples")

# Load classes
classes_file = DATA_SOURCE_DIR / "classes.txt"
if classes_file.exists():
    with open(classes_file, 'r') as f:
        classes = [line.strip() for line in f.readlines()]
    print(f"Classes: {classes}")
else:
    classes = ["roof"]
    print(f"No classes.txt found, using default: {classes}")

Validating 200 image-label pairs...

✓ Valid pairs: 200
✗ Invalid pairs: 0
Total dataset size: 200 samples
Classes: ['roof']


## 4. Prepare Dataset Structure

In [4]:
# Create YOLO directory structure
train_images_dir = DATASET_DIR / "images" / "train"
val_images_dir = DATASET_DIR / "images" / "val"
test_images_dir = DATASET_DIR / "images" / "test"

train_labels_dir = DATASET_DIR / "labels" / "train"
val_labels_dir = DATASET_DIR / "labels" / "val"
test_labels_dir = DATASET_DIR / "labels" / "test"

for dir_path in [train_images_dir, val_images_dir, test_images_dir, 
                  train_labels_dir, val_labels_dir, test_labels_dir]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Split dataset: 70% train, 20% val, 10% test
train_size = 0.7
val_size = 0.2
test_size = 0.1

# First split: train + temp (val + test)
train_pairs, temp_pairs = train_test_split(data_pairs, test_size=(1 - train_size), random_state=42)

# Second split: val and test
val_pairs, test_pairs = train_test_split(temp_pairs, test_size=test_size/(val_size + test_size), random_state=42)

print(f"Dataset split:")
print(f"  Train: {len(train_pairs)} ({len(train_pairs)/len(data_pairs)*100:.1f}%)")
print(f"  Val:   {len(val_pairs)} ({len(val_pairs)/len(data_pairs)*100:.1f}%)")
print(f"  Test:  {len(test_pairs)} ({len(test_pairs)/len(data_pairs)*100:.1f}%)")

# Function to copy files
def copy_dataset(pairs, images_dir, labels_dir):
    for img_path, label_path in pairs:
        shutil.copy2(img_path, images_dir / img_path.name)
        shutil.copy2(label_path, labels_dir / label_path.name)

# Copy files
print("\nCopying files...")
copy_dataset(train_pairs, train_images_dir, train_labels_dir)
copy_dataset(val_pairs, val_images_dir, val_labels_dir)
copy_dataset(test_pairs, test_images_dir, test_labels_dir)

print("✓ Dataset prepared successfully!")

Dataset split:
  Train: 139 (69.5%)
  Val:   40 (20.0%)
  Test:  21 (10.5%)

Copying files...
✓ Dataset prepared successfully!


## 5. Configure YOLO Training Parameters

In [5]:
# Create dataset.yaml configuration file
dataset_yaml = {
    'path': str(DATASET_DIR),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(classes),
    'names': {i: cls for i, cls in enumerate(classes)}
}

# Save YAML file
yaml_path = DATASET_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)

print(f"Dataset configuration saved to: {yaml_path}")
print("\nDataset YAML content:")
print(yaml.dump(dataset_yaml, default_flow_style=False))

Dataset configuration saved to: /home/jovyan/work/roof_dataset_yolo/data.yaml

Dataset YAML content:
names:
  0: roof
nc: 1
path: /home/jovyan/work/roof_dataset_yolo
test: images/test
train: images/train
val: images/val



## 6. Train YOLO Model with Fine-Tuning

In [6]:
# Training parameters
EPOCHS = 50
BATCH_SIZE = 8  # Reduced batch size for CPU training
IMG_SIZE = 640
DEVICE = 'cpu'  # Use 'cpu' for CPU or 0 for GPU device if available

# Load a pre-trained YOLO model
# Options: yolov8n (nano), yolov8s (small), yolov8m (medium), yolov8l (large), yolov8x (extra-large)
model = YOLO('yolov8n.pt')

print(f"Model loaded: YOLOv8 Nano")
print(f"\nTraining configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMG_SIZE}")
print(f"  Device: {DEVICE}")
print(f"  Dataset: {yaml_path}")

# Train the model
print("\nStarting training...")
results = model.train(
    data=str(yaml_path),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    patience=10,  # Early stopping patience
    save=True,
    save_period=5,  # Save every 5 epochs
    verbose=True,
    project='runs/roof_detection',
    name='yolov8n_finetuned',
    exist_ok=False
)

print("\n✓ Training completed!")
print(f"Results saved to: runs/roof_detection/yolov8n_finetuned")

Model loaded: YOLOv8 Nano

Training configuration:
  Epochs: 50
  Batch size: 8
  Image size: 640
  Device: cpu
  Dataset: /home/jovyan/work/roof_dataset_yolo/data.yaml

Starting training...
New https://pypi.org/project/ultralytics/8.4.9 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.8 🚀 Python-3.11.6 torch-2.9.1+cu128 CPU (13th Gen Intel Core i7-1355U)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/jovyan/work/roof_dataset_yolo/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8

## 7. Evaluate Model Performance

In [9]:
# Load the best trained model
best_model_path = 'runs/detect/runs/roof_detection/yolov8n_finetuned/weights/best.pt'
best_model = YOLO(best_model_path)

# Evaluate on validation set
print("Evaluating on validation set...")
val_results = best_model.val()

print("\nValidation Results:")
print(f"  mAP50: {val_results.box.map50:.4f}")
print(f"  mAP50-95: {val_results.box.map:.4f}")

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = best_model.val(data=str(yaml_path), split='test')

print("\nTest Results:")
print(f"  mAP50: {test_results.box.map50:.4f}")
print(f"  mAP50-95: {test_results.box.map:.4f}")

# Display training results
print("\n" + "="*50)
print("TRAINING SUMMARY")
print("="*50)
results_csv = Path('runs/detect/runs/roof_detection/yolov8n_finetuned/results.csv')
if results_csv.exists():
    results_df = pd.read_csv(results_csv)
    print(f"\nTraining history (last 10 epochs):")
    print(results_df.tail(10))

Evaluating on validation set...
Ultralytics 8.4.8 🚀 Python-3.11.6 torch-2.9.1+cu128 CPU (13th Gen Intel Core i7-1355U)
Model summary (fused): 73 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 3.0±0.5 ms, read: 30.5±3.1 MB/s, size: 261.8 KB)
val: Scanning /home/jovyan/work/roof_dataset_yolo/labels/val.cache... 40 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 40/40 3.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.5s/it 4.4s3.1ss
                   all         40        164      0.828      0.707      0.813      0.527
Speed: 1.5ms preprocess, 78.9ms inference, 0.0ms loss, 3.6ms postprocess per image
Results saved to /home/jovyan/work/runs/detect/val3

Validation Results:
  mAP50: 0.8130
  mAP50-95: 0.5266

Evaluating on test set...
Ultralytics 8.4.8 🚀 Python-3.11.6 torch-2.9.1+cu128 CPU (13th Gen Intel Core i7-1355U)
val: Fast image access ✅ (ping: 4.0±1.1 ms, read: 2

## 8. Test Predictions on New Images

In [10]:
# Run inference on test images
print("Running inference on test images...")

test_image_dir = DATASET_DIR / 'images' / 'test'
test_images = list(test_image_dir.glob('*.png'))

# Predict on a sample of test images
sample_size = min(12, len(test_images))
sample_images = test_images[:sample_size]

print(f"Found {len(test_images)} test images, running inference on {sample_size} samples...")

if len(sample_images) > 0:
    results = best_model.predict(
        source=[str(img) for img in sample_images],
        conf=0.25,
        save=True,
        project='runs/detect/runs',
        name='roof_predictions'
    )
    
    print(f"\n✓ Predictions saved to: runs/detect/runs/roof_predictions")
    
    # Visualize predictions
    fig, axes = plt.subplots(3, 4, figsize=(16, 12))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(sample_images[:12]):
        ax = axes[idx]
        img = cv2.imread(str(img_path))
        if img is not None:
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img_rgb)
        ax.set_title(f'{img_path.name}')
        ax.axis('off')
    
    plt.tight_layout()
    plt.savefig(DATASET_DIR / 'test_samples.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Inference completed!")
else:
    print("⚠️  No test images found!")

Running inference on test images...
Found 21 test images, running inference on 12 samples...

0: 640x640 4 roofs, 86.4ms
1: 640x640 1 roof, 86.4ms
2: 640x640 2 roofs, 86.4ms
3: 640x640 4 roofs, 86.4ms
4: 640x640 7 roofs, 86.4ms
5: 640x640 5 roofs, 86.4ms
6: 640x640 7 roofs, 86.4ms
7: 640x640 2 roofs, 86.4ms
8: 640x640 3 roofs, 86.4ms
9: 640x640 9 roofs, 86.4ms
10: 640x640 5 roofs, 86.4ms
11: 640x640 4 roofs, 86.4ms
Speed: 5.7ms preprocess, 86.4ms inference, 0.6ms postprocess per image at shape (1, 3, 640, 640)
Results saved to /home/jovyan/work/runs/detect/runs/detect/runs/roof_predictions

✓ Predictions saved to: runs/detect/runs/roof_predictions


<Figure size 1600x1200 with 12 Axes>


✓ Inference completed!


## 9. Export and Save Model

In [11]:
# Save the best model to different formats
output_dir = DATASET_DIR / 'trained_models'
output_dir.mkdir(exist_ok=True)

# Copy trained model if it exists
trained_model_path = Path('runs/detect/runs/roof_detection/yolov8n_finetuned/weights/best.pt')
if trained_model_path.exists():
    shutil.copy2(trained_model_path, output_dir / 'best.pt')
    print(f"✓ Copied trained model to: {output_dir / 'best.pt'}")
else:
    print("⚠️  Trained model not found, skipping copy.")

# Export to ONNX format (for deployment)
print("\nExporting model to ONNX format...")
try:
    onnx_path = best_model.export(format='onnx', imgsz=IMG_SIZE)
    print(f"✓ ONNX model saved: {onnx_path}")
except Exception as e:
    print(f"⚠️  ONNX export failed: {e}")

# Export to TorchScript format
print("\nExporting model to TorchScript format...")
try:
    torchscript_path = best_model.export(format='torchscript', imgsz=IMG_SIZE)
    print(f"✓ TorchScript model saved: {torchscript_path}")
except Exception as e:
    print(f"⚠️  TorchScript export failed: {e}")

print(f"\n✓ All model formats saved to: {output_dir}")

# Create a summary file
summary_metrics = ""
if val_results is not None:
    summary_metrics += f"""
## Training Results
- Validation mAP50: {val_results.box.map50:.4f}
- Validation mAP50-95: {val_results.box.map:.4f}"""
else:
    summary_metrics += "\n## Training Results\n- No validation metrics available"

if test_results is not None:
    summary_metrics += f"""
- Test mAP50: {test_results.box.map50:.4f}
- Test mAP50-95: {test_results.box.map:.4f}"""

summary = f"""
# YOLO Roof Detection Model - Training Summary

## Dataset Statistics
- Total samples: {len(data_pairs)}
- Training samples: {len(train_pairs)}
- Validation samples: {len(val_pairs)}
- Test samples: {len(test_pairs)}

## Model Configuration
- Model: YOLOv8 Nano
- Input size: {IMG_SIZE}x{IMG_SIZE}
- Classes: {', '.join(classes)}
- Number of classes: {len(classes)}

## Training Parameters
- Epochs: {EPOCHS}
- Batch size: {BATCH_SIZE}
- Device: {'GPU' if DEVICE != 'cpu' else 'CPU'}

## Model Locations
- PyTorch: {output_dir / 'best.pt'}
- ONNX: {output_dir / 'best.onnx'}
- TorchScript: {output_dir / 'best.torchscript.pt'}
{summary_metrics}

## Dataset Path
{DATASET_DIR}

## Training Runs Path
runs/detect/runs/roof_detection/yolov8n_finetuned
"""

summary_path = output_dir / 'TRAINING_SUMMARY.md'
with open(summary_path, 'w') as f:
    f.write(summary)

print(f"\n✓ Training summary saved to: {summary_path}")
print("\n" + "="*60)
print("PROCESS COMPLETE!")
print("="*60)

✓ Copied trained model to: /home/jovyan/work/roof_dataset_yolo/trained_models/best.pt

Exporting model to ONNX format...
Ultralytics 8.4.8 🚀 Python-3.11.6 torch-2.9.1+cu128 CPU (13th Gen Intel Core i7-1355U)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/

PyTorch: starting from 'runs/detect/runs/roof_detection/yolov8n_finetuned/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxslim>=0.1.71', 'onnxruntime'] not found, attempting AutoUpdate...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/17.5 MB ? eta -:--:--
   ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.4/17.5 MB 10.9 MB/s eta 0:00:02
   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/17.5 MB 21.3 MB/s eta 0:00:01
   ━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/17.5 MB 24.3 MB/s eta 0:00:01
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━